# ABPT TinyStories BPE compare — upload only this notebook and run

Этот notebook сам восстанавливает нужный snapshot проекта, ставит минимальные зависимости и запускает сравнение **baseline vs anchor** на **TinyStories с BPE/subword токенизацией**.

В конце он сразу строит **готовый compare-report с выводом**.

Никаких дополнительных файлов загружать не нужно.

In [ ]:
# @title 1. Restore embedded project snapshot
import base64, io, zipfile
from pathlib import Path

PAYLOAD = """..."""
repo = Path('/content/ABPT_tinystories_compare')
repo.mkdir(parents=True, exist_ok=True)
raw = base64.b64decode(PAYLOAD.encode('ascii'))
with zipfile.ZipFile(io.BytesIO(raw), 'r') as zf:
    zf.extractall(repo)
(repo / 'archive').mkdir(parents=True, exist_ok=True)
(repo / 'docs' / 'research').mkdir(parents=True, exist_ok=True)
(repo / 'data_cache').mkdir(parents=True, exist_ok=True)
%cd /content/ABPT_tinystories_compare
print('Restored repo to', repo)


In [ ]:
# @title 2. Install minimal dependencies
!pip -q install einops huggingface_hub tokenizers


In [ ]:
# @title 3. Runtime check
import json, torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(json.dumps({
    'device': DEVICE,
    'torch': torch.__version__,
    'cuda_devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}, indent=2))


In [ ]:
# @title 4. Experiment config
PRESET = 'toy'
TINYSTORIES_REPO = 'roneneldan/TinyStories'
TINYSTORIES_BYTES = 20_000_000
TINYSTORIES_VOCAB = 4096
STEPS = 5000
BATCH_SIZE = 16
SEQ_LEN = 128
EVAL_INTERVAL = 1000
print({
    'preset': PRESET,
    'tinystories_repo': TINYSTORIES_REPO,
    'tinystories_bytes': TINYSTORIES_BYTES,
    'tinystories_vocab': TINYSTORIES_VOCAB,
    'steps': STEPS,
    'batch_size': BATCH_SIZE,
    'seq_len': SEQ_LEN,
    'eval_interval': EVAL_INTERVAL,
})


In [ ]:
# @title 5. Baseline run (no anchors)
import subprocess, shlex
baseline_history = '/content/ABPT_tinystories_compare/archive/tinystories_bpe_baseline_history.json'
baseline_report = '/content/ABPT_tinystories_compare/docs/research/tinystories_bpe_baseline_report.md'
cmd = [
    'python', 'train.py',
    '--preset', PRESET,
    '--stage', 'a',
    '--dataset', 'tinystories-bpe',
    '--device', DEVICE,
    '--steps', str(STEPS),
    '--batch_size', str(BATCH_SIZE),
    '--seq_len', str(SEQ_LEN),
    '--eval_interval', str(EVAL_INTERVAL),
    '--tinystories_repo', TINYSTORIES_REPO,
    '--tinystories_bytes', str(TINYSTORIES_BYTES),
    '--tinystories_vocab_size', str(TINYSTORIES_VOCAB),
    '--history_path', baseline_history,
]
print('Running:', ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
subprocess.run([
    'python', 'scripts/generate_anchor_training_report.py',
    baseline_history, '--output', baseline_report,
], check=True)
print(baseline_report)


In [ ]:
# @title 6. Anchor run
import subprocess, shlex
anchor_history = '/content/ABPT_tinystories_compare/archive/tinystories_bpe_anchor_history.json'
anchor_report = '/content/ABPT_tinystories_compare/docs/research/tinystories_bpe_anchor_report.md'
cmd = [
    'python', 'train.py',
    '--preset', PRESET,
    '--stage', 'anchor',
    '--dataset', 'tinystories-bpe',
    '--device', DEVICE,
    '--steps', str(STEPS),
    '--batch_size', str(BATCH_SIZE),
    '--seq_len', str(SEQ_LEN),
    '--eval_interval', str(EVAL_INTERVAL),
    '--tinystories_repo', TINYSTORIES_REPO,
    '--tinystories_bytes', str(TINYSTORIES_BYTES),
    '--tinystories_vocab_size', str(TINYSTORIES_VOCAB),
    '--history_path', anchor_history,
]
print('Running:', ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
subprocess.run([
    'python', 'scripts/generate_anchor_training_report.py',
    anchor_history, '--output', anchor_report,
], check=True)
print(anchor_report)


In [ ]:
# @title 7. Build final compare report with verdict
import subprocess
compare_report = '/content/ABPT_tinystories_compare/docs/research/tinystories_bpe_compare_report.md'
subprocess.run([
    'python', 'scripts/generate_anchor_training_compare.py',
    '/content/ABPT_tinystories_compare/archive/tinystories_bpe_baseline_history.json',
    '/content/ABPT_tinystories_compare/archive/tinystories_bpe_anchor_history.json',
    '--output', compare_report,
], check=True)
print(compare_report)


In [ ]:
# @title 8. Print final verdict immediately
from pathlib import Path
compare_path = Path('/content/ABPT_tinystories_compare/docs/research/tinystories_bpe_compare_report.md')
print(compare_path.read_text(encoding='utf-8'))
